# 02. Reproduksi Baseline Model (Tanpa Class Weighting)

Notebook ini melatih model baseline **Logistic Regression** dan **Random Forest** (`class_weight=None`):
- Menggunakan `src/preprocessing.py` untuk pembersihan, 3-way stratified split (60% Train, 20% Val, 20% Test, `random_state=42`), dan `Pipeline` (`SimpleImputer` + `StandardScaler`) yang hanya di-fit pada data Train.
- Menggunakan `src/evaluation.py` untuk menghitung PR-AUC, Precision, Recall, F1, dan `Recall@20% LOC` pada **Validation dan Test** serta membandingkannya dengan **Random Baseline (0.20)**.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sys.path.append(str(Path("..").resolve()))
from src.preprocessing import load_nasa_arff, clean_data, split_train_val_test, build_model_pipeline
from src.evaluation import calculate_metrics, calculate_recall_at_20_percent_loc

data_dir = Path("../data")
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

datasets = ["JM1.arff", "PC4.arff", "MC1.arff"]
results = []

for fname in datasets:
    df, target_col = load_nasa_arff(data_dir / fname)
    X, y = clean_data(df, target_col=target_col)

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, train_size=0.60, val_size=0.20, test_size=0.20, random_state=42
    )

    models = {
        "Baseline_LR": build_model_pipeline(
            LogisticRegression(class_weight=None, max_iter=1000, random_state=42), use_scaler=True
        ),
        "Baseline_RF": build_model_pipeline(
            RandomForestClassifier(n_estimators=100, class_weight=None, random_state=42), use_scaler=False
        ),
    }

    for name, model in models.items():
        model.fit(X_train, y_train)

        val_prob = model.predict_proba(X_val)[:, 1]
        val_pred = model.predict(X_val)
        val_metrics = calculate_metrics(y_val.values, val_pred, val_prob)
        val_rec20 = calculate_recall_at_20_percent_loc(y_val.values, val_prob, X_val["LOC_TOTAL"].values)

        test_prob = model.predict_proba(X_test)[:, 1]
        test_pred = model.predict(X_test)
        test_metrics = calculate_metrics(y_test.values, test_pred, test_prob)
        test_rec20 = calculate_recall_at_20_percent_loc(y_test.values, test_prob, X_test["LOC_TOTAL"].values)

        results.append({
            "Dataset": fname.replace(".arff", ""),
            "Model": name,
            "Val_PR_AUC": round(val_metrics["pr_auc"], 4),
            "Val_Precision": round(val_metrics["precision"], 4),
            "Val_Recall": round(val_metrics["recall"], 4),
            "Val_F1": round(val_metrics["f1"], 4),
            "Val_Recall20_LOC": round(val_rec20, 4),
            "Test_PR_AUC": round(test_metrics["pr_auc"], 4),
            "Test_Precision": round(test_metrics["precision"], 4),
            "Test_Recall": round(test_metrics["recall"], 4),
            "Test_F1": round(test_metrics["f1"], 4),
            "Test_Recall20_LOC": round(test_rec20, 4),
            "Random_Baseline_LOC": 0.2000
        })

res_df = pd.DataFrame(results)
res_df.to_csv(results_dir / "baseline_metrics.csv", index=False)
print(res_df.to_string(index=False))

Dataset       Model  Val_PR_AUC  Val_Precision  Val_Recall  Val_F1  Val_Recall20_LOC  Test_PR_AUC  Test_Precision  Test_Recall  Test_F1  Test_Recall20_LOC  Random_Baseline_LOC
    JM1 Baseline_LR      0.3783         0.4627      0.0928  0.1546            0.0689       0.3830          0.5741       0.0925   0.1594             0.1164                  0.2
    JM1 Baseline_RF      0.4243         0.5593      0.1976  0.2920            0.1287       0.3657          0.4919       0.1821   0.2658             0.1552                  0.2
    PC4 Baseline_LR      0.5864         0.7222      0.3611  0.4815            0.3611       0.6337          0.7727       0.4857   0.5965             0.5143                  0.2
    PC4 Baseline_RF      0.6396         0.7143      0.2778  0.4000            0.3611       0.7025          0.8000       0.4571   0.5818             0.5714                  0.2
    MC1 Baseline_LR      0.0728         0.0000      0.0000  0.0000            0.4444       0.2127          0.0000       